# COMS W2132 Intermediate Computing in Python
## Errors, Exceptions, and Debugging Basics

**Date**: April 20 2026
**Instructor**: Daniel Bauer <bauer@cs.columbia.edu> 

**Reading**:
Data Structures and Algorithms in Python, Chapter 1.7

---

## 1. Exceptions

**Exceptions** are unexpected events that may happen while the program runs. Usually (but not always) exceptions indicate an **error**, resulting either from a bug in the program, or from unexpected input. 

For example, when you try to open an file that does not exist, you will get a FileNotFoundError.

In [6]:
f = open("nonexistentfile.txt",'r') 

FileNotFoundError: [Errno 2] No such file or directory: 'nonexistentfile.txt'

Exceptions are usually fatal -- they will cause the program to crash. When that happens, the interpreter will print the exception and a **traceback**, indicating the the history of function calls (i.e. the content of the runtime stack) that resulted in the exception. It's important to be able to read and understand tracebacks as a first step in **debugging**. 

The programmer can often anticipate excepeptions (especially those resulting from unexpected inputs) and write code to **handle** them in an effort to recover.

In [7]:
try: 
    f = open("nonexistentfile.txt",'r')
except FileNotFoundError: 
    print("File not found, please enter a new filename.")

File not found, please enter a new filename.


There are also situations in which the programmer may **raise** (**throw**) their own exceptions.

### 1.1 Exception Types 

Conceptually, **Compile Time Exceptions** occur before the program even runs, while **Runtime Exceptions** happen while the program is running. In Python, which is an intepreted language with dynamic typing, the only real command compile time exception is the `SyntaxError`. In languages with static typing (such as Java), the compiler will perform type checking and will then report incompatible types. 



In [132]:
while True print('Hello world')

SyntaxError: invalid syntax (2884618176.py, line 1)

Runtime exeptions come in many different types. Exceptions are objects, i.e. instances of Exception classes. These form a class hierarchy (inheritance). Here is an overview of the most common built-in exceptions you should know (I removed some exceptions that are very rare -- for a full list of built-in exceptions, see https://docs.python.org/3/library/exceptions.html)

```
BaseException
 ├── KeyboardInterrupt
 ├── SystemExit
 └── Exception
      ├── ArithmeticError
      │    └── ZeroDivisionError
      ├── AssertionError
      ├── AttributeError
      ├── ImportError
      │    └── ModuleNotFoundError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── MemoryError
      ├── NameError
      │    └── UnboundLocalError
      ├── OSError
      │    ├── FileNotFoundError
      │    ├── PermissionError
      │    └── TimeoutError
      ├── RuntimeError
      │    ├── NotImplementedError
      │    └── RecursionError
      ├── StopIteration
      ├── SyntaxError
      │    └── IndentationError        
      ├── TypeError
      ├── ValueError
      └── Warning
```

In [133]:
while True: 
    pass
    

KeyboardInterrupt: 

You can create your own exception types by inheriting from one of the built-in exceptions. 

In [15]:
class UserDoesntUnderstandHowToUseTheProgramException(BaseException): 
    pass
    

### 1.2 Reading Tracebacks

When an exception occurs, the current function or method call is interrupted. *The exception is passed down the runtime stack, until it is either handled or reaches the top level / main portion of the program.* If the exception is not handled, it will cause the program to crash and it will print a **stack trace**, providing the history of function calls. Consider the following program.

In [134]:
def check(a,b):
    return a % b == 0

def check_factors(n):
    factors = []
    for i in range(n):
        if check(n,i):
            factors.append(i)
    return factors

def main():
    n = 24
    print(check_factors(n))

main()

ZeroDivisionError: integer modulo by zero

The stack trace is read from the bottom up. The `ZeroDivisionError` occurs in line 2, and the actual offending line inside of the `check` function is printed at the end of the traceback. The check function was called in line 7, inside of the `check_factors` function, which is printed above. `check_factors`, in turn, was called from the `main()` function (line 13), which was in tern called on the top-level of the program (line 15). 

The traceback is useful for debugging because it allows us to understand exactly under which conditions the exception occurs. The `check` function itself is perfectly fine, but we are calling it with incorrect parameters. The actual mistake we made is in `check_factors`: the for loop starts at i=0, which is passed to `check(n,i)`, causing the `ZeroDivisionError`. 
So when we debug the program, we may want to start at the offending function, and then move up the the traceback. 

**Recursion:** When an error happens in a recursive function, the traceback will list the same function multiple times. The exception is passed along all previous levels of recursive calls (which correspond to stack frames on the runtime stack).

In [135]:
def sum_list(li): 
    #if not li:
    #    return 0 
    return li[0] + sum_list(li[1:]) 

def main():
    sum_list([1,2,3,4,5])

main()

IndexError: list index out of range

### 1.3 Handling Exceptions

The `try...except...` construction is used to handle exceptions. Note: You typically don't want to do this with logic/programmer errors like the ones in the previous sections -- handling such errors would likely mask the real issue and the program would be much more difficult to debug.  
But we want to handle exceptions that are caused by faulty user input (or other context conditions, such as file or system issues), so that we can try to correct the problem without crashing the program.

In [137]:
try:
    x = int(input("Enter your age in years: "))
    print(10 / x)
except ValueError:
    print("That wasn't an integer.")

Enter your age in years:  hello


That wasn't an integer.


The exception can be handled at any level in the runtime stack.

In [140]:
def get_input():
    x = int(input("Enter your age in years: "))
    return 10 / x

def main():
    try: 
        result = get_input()
        print(result)
    except ValueError: 
        print("That wasn't an integer.")

main()

Enter your age in years:  0


ZeroDivisionError: division by zero

We can catch multiple error types:

In [142]:
try:
    x = int(input("Enter your age in years: "))
    print(10 / x)
except (ValueError, ZeroDivisionError):  # can specify multiple error types as tuple
    print("Invalid input.")

Enter your age in years:  0


Invalid input.


or handle each error type separately in multiple except blocks:

In [144]:
try:
    x = int(input("Enter your age in years: "))
    print(10 / x)
except (ValueError):
    print("That wasn't an integer.")
except (ZeroDivisionError):
    print("Age can't be 0.")
    

Enter your age in years:  hello


That wasn't an integer.


**finally and context managers**: An optional `finally` block is always run, whether an exception occurs or not. This is usually used for clean-up actions, such as closing a file. 

In [45]:
f = None
try:
    f = open("data.txt", "r")   # FileNotFoundError can be caught now
    line = f.readline()
    n = int(line)              # ValueError can be caught
    print("n =", n)
except (FileNotFoundError, ValueError) as e:
    print("Problem reading file")
finally:
    if f is not None:
        f.close()

Problem reading file


Because this is slightly messy, it's often better to use a context manager (which calls a special method on the file object making sure that the file is closed no matter what). 

In [47]:
try:
    with open("data.txt", "r") as f:   # automatically cloase the file even if an exception happens
        line = f.readline()
        n = int(line)
        print("n =", n)
except (FileNotFoundError, ValueError):
    print("Problem reading file")


Problem reading file


**Printing information about exceptions:** Even when an exception is handled, it is sometimes useful to print information about it, or the complete traceback. We can assign a name to the exception object. 

In [51]:
try:
    with open("data.txt", "r") as f:   # automatically cloase the file even if an exception happens
        line = f.readline()
        n = int(line)
        print("n =", n)
except (FileNotFoundError, ValueError) as e:
    print("Problem reading file:", e)

Problem reading file: [Errno 2] No such file or directory: 'data.txt'


In [57]:
import traceback

def f():
    return g()

def g():
    return 1 / 0

try:    
    f()
except Exception:
    traceback.print_exc()

print("Continuing the program after the exception.")

Continuing the program after the exception.


Traceback (most recent call last):
  File "/var/folders/7b/683qz2414xb4rjtstjwrhzg80000gn/T/ipykernel_87316/4147254441.py", line 10, in <module>
    f()
  File "/var/folders/7b/683qz2414xb4rjtstjwrhzg80000gn/T/ipykernel_87316/4147254441.py", line 4, in f
    return g()
           ^^^
  File "/var/folders/7b/683qz2414xb4rjtstjwrhzg80000gn/T/ipykernel_87316/4147254441.py", line 7, in g
    return 1 / 0
           ~~^~~
ZeroDivisionError: division by zero


This is often useful when you want to write errors to a log file, so you can later use the log for debugging. 


In [59]:
import traceback

def f():
    return g()

def g():
    return 1 / 0

with open("error.log", "a") as log:
    try:
        f()
    except Exception:
        # print stack trace to the log file, then keep going
        traceback.print_exc(file=log)

print("Continuing after logging the error...")
print("Program still running.")

Continuing after logging the error...
Program still running.


In [60]:
!cat error.log # This is a shell command to view the content of error.log

Traceback (most recent call last):
  File "/var/folders/7b/683qz2414xb4rjtstjwrhzg80000gn/T/ipykernel_87316/107366364.py", line 11, in <module>
    f()
  File "/var/folders/7b/683qz2414xb4rjtstjwrhzg80000gn/T/ipykernel_87316/107366364.py", line 4, in f
    return g()
           ^^^
  File "/var/folders/7b/683qz2414xb4rjtstjwrhzg80000gn/T/ipykernel_87316/107366364.py", line 7, in g
    return 1 / 0
           ~~^~~
ZeroDivisionError: division by zero


### 1.5 Exception Handling vs. Explicit Verification

Is it better to explicitly verify the input or use an exception handler? 

In [76]:
    x = input("Enter your age in years: ")
    if x.isnumeric():
        n = int(x)
        if n != 0:
            print(10 / n)
        else: 
            print("Age can't be 0")
    else: 
        print("That wasn't an integer.")    

Enter your age in years:  0


Age can't be 0


Usually exception handling makes the code easier to read. It is also more efficient because testing error conditions explicitly may be expensive.

*"It's easier to ask forgiveness than it is to get permission."* (attributed to Grace Hoppper)

### 1.5 Raising Exceptions

You can use `raise` to raise an exception. 

In [65]:
try:
    x = int(input("Enter your age in years: "))
    if x < 1: 
        raise ValueError("age must be positive.")
    print(10 / x)
except ValueError as e:
    print("That wasn't an integer:",e)
except ZeroDivisionError:
    print("Age can't be 0.")

Enter your age in years:  0


That wasn't an integer: age must be positive.


You can raise your own user-defined exception types. 

In [66]:
try:
    x = int(input("Enter your age in years: "))
    if x < 1: 
        raise UserDoesntUnderstandHowToUseTheProgram("You can't have a negative age, you moron!")
    print(10 / x)
except ValueError as e:
    print("That wasn't an integer:",e)
except ZeroDivisionError:
    print("Age can't be 0.")

Enter your age in years:  -1


UserDoesntUnderstandHowToUseTheProgram: You can't have a negative age, you moron!

**Some exceptions are not errors** instead they are used to signal certain anticipated conditions in your program. For example, recall the `StopIterationException` raised by the `next()` method in an iterator. 

## 2. Debugging Basics

*"Debugging is twice as hard as writing the code in the first place. Therefore, if you write the code as cleverly as possible, you are, by definition, not smart enough to debug it."* (Brian Kernighan)

*"Writing software is 10% programming, and 90% understanding why it doesn't work."*


* Syntax errors (including indentation errors), as well as some name-errors (unused names, references to undefined names), are often indicated by the IDE. This is convenient because you can catch the bug before even running the program. The offending line is typically underlined (e.g. VsCode) or the entire line is marked with a symbol (e.g. in Spyder). 

* Other errors are more difficult to find.
  * If your code crashes with an exception, you are lucky because the traceback will give you some information about where to start tracing down the error (this is why you don't want to maskerate logic/programming errors by handling exceptions too agrressively). 

  * The most challenging category of bugs are logic errors. The program runs and terminates normally (no exception, no traceback!), but the output is incorrect. Worse, the output may be correct in _some_ cases, but not others. 
 
* AI tools can sometimes indicate type errors and logic errors, but only if they understand what you are trying to achieve. 

## 2.1 Using print statements for debugging

A simple way to debug is to use print statements to show intermediate results. Typically, you would start from the point where the incorrect output is produced, and then you retrace your steps. You may for example ask "going into this loop, what is the value variable x refers to -- is this what I expect?"

The advantage of print statements is that they are simple to use, and they work independently of any specific IDE or debugger tool. The disadvantage is that you must delete or comment-out such debugging statements. 

In [103]:
def binary_search(a, x): # find the index for x in a
    left = 0
    right = len(a) 
    while left < right: 
        mid = (left + right) / 2
        if a[mid] == x:
            return mid
        elif a[mid] < x:
            left = mid + 1
        else:
            right = mid - 1
    return -1 # not found


In [104]:
binary_search([1,5,9,13,24,141], 9)

TypeError: list indices must be integers or slices, not float

In [105]:
def binary_search(a, x): # find the index for x in a
    left = 0
    right = len(a) 
    while left < right: 
        print(left, right) # ADDED
        mid = (left + right) / 2
        print(mid)  #ADDED 
        if a[mid] == x: 
            return mid
        elif a[mid] < x:
            left = mid + 1
        else:
            right = mid - 1
    return -1 # not found

In [106]:
binary_search([1,5,9,13,24,141], 9)

0 6
3.0


TypeError: list indices must be integers or slices, not float

In [114]:
def binary_search(a, x): # find the index for x in a
    left = 0
    right = len(a) 
    while left < right:         
        mid = (left + right) // 2 # FIXED
        if a[mid] == x: 
            return mid
        elif a[mid] < x:
            left = mid + 1
        else:
            right = mid - 1
    return -1 # not found

In [115]:
binary_search([1,5,9,13,24,141], 9) # Hmm, not correct

-1

In [119]:
def binary_search(a, x): # find the index for x in a
    left = 0
    right = len(a) 
    while left < right:      
        mid = (left + right) // 2 # FIXED
        print("left:",left, "right:", right, "mid:", mid, "a[mid]:",a[mid])
        if a[mid] == x: 
            return mid
        elif a[mid] < x:
            left = mid + 1
        else:
            right = mid - 1
    return -1 # not found

In [120]:
binary_search([1,5,9,13,24,141], 9) # Hmm, not correct

left: 0 right: 6 mid: 3 a[mid]: 13
left: 0 right: 2 mid: 1 a[mid]: 5


-1

In [121]:
def binary_search(a, x): # find the index for x in a
    left = 0
    right = len(a) 
    while left < right:      
        mid = (left + right) // 2 # FIXED
        print("left:",left, "right:", right, "mid:", mid, "a[mid]:",a[mid])
        if a[mid] == x: 
            return mid
        elif a[mid] < x:
            left = mid + 1
        else:
            right = mid - 1
        print("end of loop -- left:",left, "right:", right)

    return -1 # not found

In [122]:
binary_search([1,5,9,13,24,141], 9) # Hmm, not correct

left: 0 right: 6 mid: 3 a[mid]: 13
end of loop -- left: 0 right: 2
left: 0 right: 2 mid: 1 a[mid]: 5
end of loop -- left: 2 right: 2


-1

Aha! looks like the loop is terminating too early. 


In [123]:
def binary_search(a, x): # find the index for x in a
    left = 0
    right = len(a) 
    while left <= right:   # FIXED
        mid = (left + right) // 2 # FIXED
        print("left:",left, "right:", right, "mid:", mid, "a[mid]:",a[mid])
        if a[mid] == x: 
            return mid
        elif a[mid] < x:
            left = mid + 1
        else:
            right = mid - 1
        print("end of loop -- left:",left, "right:", right)

    return -1 # not found

In [124]:
binary_search([1,5,9,13,24,141], 9) # Hmm, not correct

left: 0 right: 6 mid: 3 a[mid]: 13
end of loop -- left: 0 right: 2
left: 0 right: 2 mid: 1 a[mid]: 5
end of loop -- left: 2 right: 2
left: 2 right: 2 mid: 2 a[mid]: 9


2

In [131]:
binary_search([1,5,9,13,24,141], 142) # How about a different test case?

left: 0 right: 6 mid: 3 a[mid]: 13
end of loop -- left: 4 right: 6
left: 4 right: 6 mid: 5 a[mid]: 141
end of loop -- left: 6 right: 6


IndexError: list index out of range

## 2.2 Using the debugger


The debugger allows us to inspect the namespace (variables and values they refer to), and call stack at a specific time. We can set a number of break-points and then have the interpreter step from breack-point to breack-point (see vscode demo). 

## 3. Testing Basics

Unit testing meanbs writing small, automated checks that verify that “unit” (typically a single function or method) of code works correctly. 

Unit tests are meant to be a permanent way of ensuring correctness. If you make a change in your code, run the tests to make sure everything is still workoing as expected. 
Tests also serve as a king of documentation, since the tests illustrate how to run the code and demonstrate corner cases. 

A good unit test should be:
* Fast (milliseconds)
* Deterministic (same result every run)
* Isolated (doesn’t depend on external resources like user input/APIs/DB/files unless you’re explicitly testing those)

Notably, unit testing will catch any exceptions, so an error in one part of the program will not prevent the tests for the other units from running.


A unit test usually consists of three steps. 
* Arrange: set up inputs 
* Act: call the function / method
* Assert: check the result


There are two popular unit testing frameworks in python: `unittest` and `pytest`. Here we focus on `unittest` because it is built into the Python standard library and has a more gentle learning curve. 


### 3.1 Python's unittest

Python's unittest module is built into the standard library. It's meant to be similar in style to other testing frameworks (e.g. JUnit for Java). 

To write a set of unit tests: 
* create a class that inherits from `unittest.TestCase`
* write methods for that class whose names start with `test_`
* use assertion methods, e.g. `self.assertEqual(...)` to make assertions about the output.




To run the unit tests, you can run `python -m unittest program.py`

In [ ]:
import unittest 

class TestBinarySearch(unittest.TestCase):
    def test_finds_item_in_middle(self):
        self.assertEqual(binary_search([1, 2, 3, 4, 5], 3), 2)

    def test_finds_item_at_start(self):
        self.assertEqual(binary_search([1, 2, 3, 4, 5], 1), 0)

    def test_finds_item_at_end(self):
        self.assertEqual(binary_search([1, 2, 3, 4, 5], 5), 4)

    def test_returns_minus_one_when_not_found(self):
        self.assertEqual(binary_search([1, 2, 3, 4, 5], 99), -1)

    def test_empty_list_returns_minus_one(self):
        self.assertEqual(binary_search([], 1), -1)

    def test_singleton_found(self):
        self.assertEqual(binary_search([10], 10), 0)

    def test_singleton_not_found(self):
        self.assertEqual(binary_search([10], 5), -1)

    def test_duplicate_values_returns_an_index_with_matching_value(self):
        a = [1, 2, 2, 2, 3]
        idx = binary_search(a, 2)
        # For duplicates, any correct index is acceptable as long as it points to a 2
        self.assertNotEqual(idx, -1)
        self.assertEqual(a[idx], 2)



In [ ]:
Sometimes, it makes sense to create a common setup before running the methods. 

class TestBinarySearch(unittest.TestCase):

    def setUp(self):  # Run before every test method
        self.test_list = [1,2,3,4,5]

    def tearDown(self): # run after every test method
        self.test_list = None

    def test_finds_item_in_middle(self):
        self.assertEqual(binary_search([1, 2, 3, 4, 5], 3), 2)

    def test_finds_item_at_start(self):
        self.assertEqual(binary_search([1, 2, 3, 4, 5], 1), 0)

    def test_finds_item_at_end(self):
        self.assertEqual(binary_search([1, 2, 3, 4, 5], 5), 4)

    def test_returns_minus_one_when_not_found(self):
        self.assertEqual(binary_search([1, 2, 3, 4, 5], 99), -1)

    def test_empty_list_returns_minus_one(self):
        self.assertEqual(binary_search([], 1), -1)

    def test_singleton_found(self):
        self.assertEqual(binary_search([10], 10), 0)

    def test_singleton_not_found(self):
        self.assertEqual(binary_search([10], 5), -1)

    def test_duplicate_values_returns_an_index_with_matching_value(self):
        a = [1, 2, 2, 2, 3]
        idx = binary_search(a, 2)
        # For duplicates, any correct index is acceptable as long as it points to a 2
        self.assertNotEqual(idx, -1)
        self.assertEqual(a[idx], 2)

**Common Assertions**

  
* self.assertEqual(a, b)
* self.assertNotEqual(a, b)
* self.assertTrue(x) / self.assertFalse(x)
* self.assertIsNone(x) / self.assertIsNotNone(x)
* self.assertIn(item, collection)
* with self.assertRaises(SomeError): ...
